# Basketball Action Recognition and Performance Analytics

This notebook is my implementation of automated basketball video analysis. It takes a basketball clip as input and produces structured per player statistics. hardware.

## Pipeline overview

The pipeline is five stages:

1. **Video ingestion** (Section 4). Download a YouTube clip and trim it to a working window.
2. **Player detection and tracking** (Sections 5-6, 10). YOLOv8 fine tuned on SportsMOT, paired with ByteTrack for multi object tracking.
3. **Ball trajectory** (Section 14). YOLOv8 specialist or COCO fallback, then smoothed by a 2D constant velocity Kalman filter.
4. **Analytics** (Sections 15-18). Shot attempt detection, ball possession, team classification.
5. **Evaluation and visualisation** (Sections 19-23). HOTA/MOTA/IDF1 against SportsMOT validation, shot chart, dashboard.

All configuration lives in the PipelineConfig dataclass in Section 1. Trained model weights are stored on Google Drive under CONFIG.work_dir/models/.

## How to run

1. Runtime, Change runtime type, T4 GPU.
2. Run the Drive mount cell below.
3. Run Sections 1-3 to set up CONFIG, imports, and utilities.
4. Run the remaining sections in order.

## Datasets used

- SportsMOT (Cui et al., 2023) for player detection and tracking. 15 basketball training sequences, 15 basketball validation sequences, around 12,384 training frames after format conversion.
- DeepSportradar v1 (Van Zandycke et al., 2022) intended as ball detector training data. See Section 4.4.1 of the dissertation for why this fell back to COCO sports_ball.
- A single 12 second Duke vs Baylor 2025 NCAA broadcast clip for inference demos.


## Mount Google Drive

Mounting Drive lets the notebook persist downloads, trained models, and outputs across Colab sessions. Without this, everything is lost on runtime disconnect.


In [ ]:
# Mount Google Drive. This must be the first cell run.
# If it errors with "Mountpoint must not already contain files",
# uncomment the shutil.rmtree line below.

# import shutil; shutil.rmtree('/content/drive', ignore_errors=True)

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# Verify the mount worked
import os
drive_root = "/content/drive/MyDrive"
if os.path.isdir(drive_root):
    print(f"Drive mounted at {drive_root}")
else:
    print("Drive mount failed; check the auth popup did not time out")


---
# Part A, Foundation

## Section 1: Global Configuration
All pipeline parameters in one place. Edit, re-run, downstream cells use new values.


In [ ]:
# ----------------------------------------------------------------------
# GLOBAL CONFIG
# ----------------------------------------------------------------------
from dataclasses import dataclass, field, asdict
from pathlib import Path
import json
from typing import List

@dataclass
class PipelineConfig:
    # ── Video source ────────────────────────────────────────────────────
    youtube_url: str = "https://www.youtube.com/watch?v=N-eE7ZwTuno"                  # set this to your test video URL
    manual_video_path: str = ""
    trim_start_sec: int = 248
    trim_duration_sec: int = 12
    fps: int = 30                          # auto-detected after ingestion

    # ── Models ──────────────────────────────────────────────────────────
    player_model: str = "yolov8n.pt"       # generic fallback
    ball_model:   str = "yolov8m.pt"       # generic fallback
    player_weights: str = ""               # set after Section 10
    ball_weights:   str = ""               # set after Section 13

    # ── Detection thresholds ────────────────────────────────────────────
    player_conf: float = 0.40
    ball_conf:   float = 0.20
    iou_threshold: float = 0.45
    ball_imgsz: int = 1280

    # ── Tracking ────────────────────────────────────────────────────────
    tracker_config: str = "bytetrack.yaml"

    # ── Shot detection ──────────────────────────────────────────────────
    upward_velocity_threshold: float = -2.0
    min_upward_frames: int = 2
    shot_dedup_window: int = 30

    # ── Possession ──────────────────────────────────────────────────────
    possession_distance_px: float = 80.0
    possession_smoothing_window: int = 5

    # ── SportsMOT ───────────────────────────────────────────────────────
    sportsmot_basketball_only: bool = True       # only basketball sequences
    sportsmot_max_train_seqs:  int  = 20         # cap to keep training feasible
    sportsmot_max_val_seqs:    int  = 10

    # ── DeepSportradar ──────────────────────────────────────────────────
    deepsport_dataset_id: str = "deepsportradar/basketball-instants-dataset"

    # ── Training ────────────────────────────────────────────────────────
    train_epochs: int = 15
    train_imgsz_player: int = 640
    train_imgsz_ball: int = 1280
    train_batch_player: int = 16
    train_batch_ball: int = 4               # smaller because imgsz is bigger
    train_patience: int = 8

    # ── Reproducibility ─────────────────────────────────────────────────
    random_seed: int = 42

    # ── Paths ───────────────────────────────────────────────────────────
    work_dir: str = "/content/drive/MyDrive/basketball_ai"

    def __post_init__(self):
        self.work_dir = str(Path(self.work_dir).absolute())
        for sub in ["", "output", "models", "logs", "data",
                    "data/sportsmot", "data/deepsport",
                    "data/sportsmot_yolo", "data/deepsport_yolo"]:
            Path(self.work_dir, sub).mkdir(parents=True, exist_ok=True)

    def save(self, path=None):
        path = path or Path(self.work_dir) / "logs" / "config.json"
        with open(path, "w") as f:
            json.dump(asdict(self), f, indent=2)
        return path

CONFIG = PipelineConfig()
CONFIG.save()
print(" Config initialised.")
print(f"   Working dir: {CONFIG.work_dir}")

## Section 2: Environment Setup


In [ ]:
!nvidia-smi
!pip install -q ultralytics opencv-python-headless pandas matplotlib \
                yt-dlp scikit-learn roboflow filterpy streamlit pyngrok \
                kaggle huggingface_hub motmetrics tqdm
print("\n Packages installed.")

In [ ]:
import os, sys, glob, logging, warnings, random, shutil, zipfile
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from collections import defaultdict, deque
from typing import Optional, Tuple, List, Dict
from tqdm.auto import tqdm
from IPython.display import Video, display, HTML, Image as IPyImage

from ultralytics import YOLO
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from filterpy.kalman import KalmanFilter

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

random.seed(CONFIG.random_seed)
np.random.seed(CONFIG.random_seed)
print(" Imports successful.")

## Section 3: Utilities & Logging


In [ ]:
from datetime import datetime

def setup_logger(name: str = "basketball") -> logging.Logger:
    """Logger that writes to console + timestamped file."""
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    fmt = logging.Formatter("%(asctime)s │ %(levelname)-7s │ %(message)s",
                            datefmt="%H:%M:%S")
    ch = logging.StreamHandler(); ch.setFormatter(fmt); logger.addHandler(ch)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = Path(CONFIG.work_dir) / "logs" / f"run_{timestamp}.log"
    fh = logging.FileHandler(log_file); fh.setFormatter(fmt); logger.addHandler(fh)
    logger.info(f"Logger initialised → {log_file}")
    return logger

log = setup_logger()

In [ ]:
def video_info(path: str) -> dict:
    if not Path(path).exists():
        raise FileNotFoundError(f"Video not found: {path}")
    cap = cv2.VideoCapture(path)
    info = {
        "path": path,
        "fps":    cap.get(cv2.CAP_PROP_FPS),
        "frames": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
        "width":  int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    }
    info["duration_sec"] = info["frames"] / info["fps"] if info["fps"] else 0
    cap.release()
    return info

def show_video(path: str, width: int = 720):
    if Path(path).exists():
        print(f" {path}")
        display(Video(path, embed=True, width=width))
    else:
        log.warning(f"Video missing: {path}")

def output_path(*parts) -> Path:
    p = Path(CONFIG.work_dir, "output", *parts)
    p.parent.mkdir(parents=True, exist_ok=True)
    return p

def safe_div(a, b, default=0.0):
    return a / b if b else default

log.info("Utilities loaded.")

---
# Part B, Core Pipeline

> This part runs with generic pretrained YOLOv8. Once Part C is done, the same code transparently switches to specialist trained models because `run_detection()` and `run_tracking()` use `CONFIG.player_weights` if set.

## Section 4: Video Ingestion


In [ ]:
def acquire_video() -> str:
    raw_path = Path(CONFIG.work_dir, "data", "raw.mp4")
    trimmed_path = Path(CONFIG.work_dir, "data", "clip.mp4")

    if CONFIG.manual_video_path and Path(CONFIG.manual_video_path).exists():
        log.info(f"Using uploaded video: {CONFIG.manual_video_path}")
        shutil.copy(CONFIG.manual_video_path, raw_path)
    elif CONFIG.youtube_url:
        log.info(f"Downloading: {CONFIG.youtube_url}")
        cmd = (
            f'yt-dlp -f "bestvideo[ext=mp4][height<=720]+bestaudio[ext=m4a]/'
            f'best[ext=mp4][height<=720]" --merge-output-format mp4 '
            f'-o "{raw_path}" "{CONFIG.youtube_url}" --quiet'
        )
        os.system(cmd)
    else:
        raise ValueError("Set CONFIG.youtube_url OR CONFIG.manual_video_path.")

    if not raw_path.exists():
        raise RuntimeError("Video acquisition failed.")

    os.system(
        f'ffmpeg -y -ss {CONFIG.trim_start_sec} -t {CONFIG.trim_duration_sec} '
        f'-i "{raw_path}" -c:v libx264 -crf 23 -preset fast -an '
        f'"{trimmed_path}" -loglevel error'
    )
    final = trimmed_path if trimmed_path.exists() else raw_path

    info = video_info(str(final))
    CONFIG.fps = int(round(info["fps"])); CONFIG.save()
    log.info(f"Video: {info['width']}x{info['height']} @ {info['fps']:.1f}fps, "
             f"{info['duration_sec']:.1f}s ({info['frames']} frames)")
    return str(final)

VIDEO = acquire_video()

In [ ]:
def preview_frame(video_path: str, frame_idx: int = 0):
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read(); cap.release()
    if ok:
        plt.figure(figsize=(12, 6))
        plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        plt.title(f"Frame {frame_idx}"); plt.axis("off")
        plt.tight_layout(); plt.show()
preview_frame(VIDEO, 0)

## Section 5: Player Detection (YOLOv8)


In [ ]:
def run_detection(video: str, model_path: str = None) -> pd.DataFrame:
    model_path = model_path or CONFIG.player_weights or CONFIG.player_model
    log.info(f"Detector: {model_path}")
    model = YOLO(model_path)
    # Find the player class index
    if CONFIG.player_weights and Path(CONFIG.player_weights).exists():
        player_cls = next((k for k, v in model.names.items()
                           if "player" in v.lower() or "person" in v.lower()), 0)
    else:
        player_cls = 0  # COCO person

    results = model.predict(
        source=video, classes=[player_cls],
        conf=CONFIG.player_conf, iou=CONFIG.iou_threshold,
        save=True, project=str(output_path("detection")),
        name="run", exist_ok=True, stream=True, verbose=False
    )
    rows = []
    for f_idx, r in enumerate(results):
        if r.boxes is None: continue
        for box, conf, cls in zip(r.boxes.xyxy, r.boxes.conf, r.boxes.cls):
            x1,y1,x2,y2 = box.tolist()
            rows.append({
                "frame": f_idx, "class_id": int(cls.item()),
                "x1":x1, "y1":y1, "x2":x2, "y2":y2,
                "cx":(x1+x2)/2, "cy":(y1+y2)/2,
                "w":x2-x1, "h":y2-y1,
                "confidence": float(conf.item())
            })
    df = pd.DataFrame(rows)
    df.to_csv(output_path("detection","detections.csv"), index=False)
    log.info(f"Detections: {len(df)} boxes in "
             f"{df['frame'].nunique() if len(df) else 0} frames")
    return df

DETECTIONS = run_detection(VIDEO)
DETECTIONS.head()

In [ ]:
dvids = glob.glob(str(output_path("detection") / "**/*.mp4"), recursive=True)
if dvids: show_video(dvids[0])

## Section 6: Multi-Object Tracking (ByteTrack)


In [ ]:
def run_tracking(video: str, model_path: str = None) -> pd.DataFrame:
    model_path = model_path or CONFIG.player_weights or CONFIG.player_model
    log.info(f"Tracker model: {model_path}")
    model = YOLO(model_path)
    if CONFIG.player_weights and Path(CONFIG.player_weights).exists():
        player_cls = next((k for k, v in model.names.items()
                           if "player" in v.lower() or "person" in v.lower()), 0)
    else:
        player_cls = 0
    results = model.track(
        source=video, classes=[player_cls],
        conf=CONFIG.player_conf, iou=CONFIG.iou_threshold,
        tracker=CONFIG.tracker_config, persist=True,
        save=True, project=str(output_path("tracking")),
        name="run", exist_ok=True, stream=True, verbose=False
    )
    rows = []
    for f_idx, r in enumerate(results):
        if r.boxes is None or r.boxes.id is None: continue
        for box, tid, conf in zip(r.boxes.xyxy, r.boxes.id, r.boxes.conf):
            x1,y1,x2,y2 = box.tolist()
            rows.append({
                "frame": f_idx, "track_id": int(tid.item()),
                "x1":x1, "y1":y1, "x2":x2, "y2":y2,
                "cx":(x1+x2)/2, "cy":(y1+y2)/2,
                "foot_x":(x1+x2)/2, "foot_y":y2,
                "confidence": float(conf.item())
            })
    df = pd.DataFrame(rows)
    df.to_csv(output_path("tracking","tracks.csv"), index=False)
    log.info(f"Tracks: {df['track_id'].nunique()} unique IDs, {len(df)} detections")
    return df

TRACKS = run_tracking(VIDEO)
TRACKS.head()

In [ ]:
tvids = glob.glob(str(output_path("tracking") / "**/*.mp4"), recursive=True)
if tvids: show_video(tvids[0])

---
# Part C, Custom Training (academic datasets)

## Section 7: Authentication

Both datasets require credentials:
- **SportsMOT** (HuggingFace): no token needed for public datasets, but a token avoids rate limits
- **DeepSportradar** (Kaggle): requires a Kaggle API token

### Setting up credentials in Colab

**Option A: Colab Secrets (recommended)**
1. Click the ** key icon** in the left sidebar
2. Add: `KAGGLE_USERNAME`, `KAGGLE_KEY`, `HF_TOKEN` (HF token optional)
3. Toggle "Notebook access" on for each
4. Run the cell below

**Option B: Manual upload**
- Download `kaggle.json` from kaggle.com → Account → Create New API Token
- The cell below will prompt you to upload it


In [ ]:
# ----------------------------------------------------------------------
# Authentication
# ----------------------------------------------------------------------
def setup_kaggle():
    """Authenticate Kaggle CLI via Colab Secrets or manual upload."""
    try:
        from google.colab import userdata
        try:
            os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
            os.environ["KAGGLE_KEY"]      = userdata.get("KAGGLE_KEY")
            log.info(" Kaggle credentials loaded from Colab Secrets")
            return True
        except Exception:
            log.info("Colab Secrets not set, falling back to manual upload")
    except ImportError:
        log.info("Not in Colab, assuming kaggle.json is in ~/.kaggle/")
        return Path.home().joinpath(".kaggle", "kaggle.json").exists()

    # Manual upload fallback
    try:
        from google.colab import files
        print("Upload your kaggle.json file...")
        uploaded = files.upload()
        if "kaggle.json" in uploaded:
            kaggle_dir = Path.home() / ".kaggle"
            kaggle_dir.mkdir(exist_ok=True)
            with open(kaggle_dir / "kaggle.json", "wb") as f:
                f.write(uploaded["kaggle.json"])
            os.chmod(kaggle_dir / "kaggle.json", 0o600)
            log.info(" kaggle.json uploaded")
            return True
    except Exception as e:
        log.error(f"Kaggle setup failed: {e}")
    return False

def setup_huggingface():
    """Authenticate HuggingFace via Colab Secrets (optional)."""
    try:
        from google.colab import userdata
        try:
            token = userdata.get("HF_TOKEN")
            os.environ["HF_TOKEN"] = token
            log.info(" HuggingFace token loaded")
            return True
        except Exception:
            log.info("HF_TOKEN not set, public dataset access only "
                     "(may hit rate limits)")
    except ImportError:
        pass
    return False

KAGGLE_OK = setup_kaggle()
HF_OK     = setup_huggingface()

## Section 8: Download SportsMOT (basketball sequences only)

**About SportsMOT** (Cui et al., ICCV 2023): 240 video sequences across basketball, football, and volleyball, with 1.6M+ bounding boxes and consistent player IDs across frames.

The full dataset is ~36 GB.
This cell uses HuggingFace's `allow_patterns` feature to download **only** the basketball sequences we actually train on. Download size: 5-10 GB depending on sequence lengths.


In [ ]:
# ----------------------------------------------------------------------
# Filter to basketball-only sequences
# ----------------------------------------------------------------------
def list_basketball_sequences(sportsmot_dir: Path) -> dict:
    """Return {split: [sequence_names]} for basketball subset."""
    splits = {}
    bk_file = sportsmot_dir / "splits_txt" / "basketball.txt"
    if not bk_file.exists():
        log.warning(f"basketball.txt not found at {bk_file}, using all sequences.")
        return {}
    bk_seqs = set(bk_file.read_text().strip().split("\n"))
    for split in ["train", "val", "test"]:
        split_dir = sportsmot_dir / "dataset" / split
        if not split_dir.exists():
            continue
        seqs = [d.name for d in split_dir.iterdir() if d.is_dir()]
        splits[split] = sorted(s for s in seqs if s in bk_seqs)
    return splits

BB_SEQS = list_basketball_sequences(SPORTSMOT_DIR)
for split, seqs in BB_SEQS.items():
    log.info(f"  {split}: {len(seqs)} basketball sequences")

In [ ]:
from pathlib import Path

target = Path("/content/drive/MyDrive/basketball_ai/data/sportsmot")
bk_seqs = set((target / "splits_txt" / "basketball.txt").read_text().strip().split("\n"))

for split in ["train", "val"]:
    d = target / "dataset" / split
    if d.exists():
        all_seqs = [s for s in d.iterdir() if s.is_dir()]
        bk_populated = [s for s in all_seqs if s.name in bk_seqs and any(s.iterdir())]
        empty = [s for s in all_seqs if not any(s.iterdir())]
        print(f"{split}: {len(bk_populated)} basketball populated, {len(empty)} empty, {len(all_seqs)} total")
        if bk_populated:
            sample = bk_populated[0]
            img_count = len(list((sample / "img1").glob("*.jpg"))) if (sample / "img1").exists() else 0
            gt_exists = (sample / "gt" / "gt.txt").exists()
            print(f"   First seq {sample.name}: {img_count} images, gt.txt={'' if gt_exists else ''}")

## Section 9: Convert SportsMOT → YOLO Format

The converter walks every sequence, reads `gt.txt`, splits it into per-frame YOLO labels, and lays out the directory structure YOLO expects


In [ ]:
# ----------------------------------------------------------------------
# Converter: SportsMOT MOT17 format → YOLOv8 format
# ----------------------------------------------------------------------
def parse_seqinfo(seq_dir: Path) -> dict:
    """Parse seqinfo.ini for image dimensions, FPS, etc."""
    info = {}
    ini_path = seq_dir / "seqinfo.ini"
    if not ini_path.exists():
        return info
    for line in ini_path.read_text().splitlines():
        if "=" in line:
            k, v = line.split("=", 1)
            info[k.strip()] = v.strip()
    return info

def convert_sportsmot_to_yolo(
    sportsmot_dir: Path,
    out_dir: Path,
    splits: dict,
    max_train: int = 20,
    max_val: int = 10
) -> Path:
    """
    Build a YOLO-formatted basketball player dataset.
    Returns path to the data.yaml config.
    """
    if (out_dir / "data.yaml").exists():
        log.info(f"YOLO dataset already exists at {out_dir}")
        return out_dir / "data.yaml"

    for s in ["train", "val"]:
        (out_dir / "images" / s).mkdir(parents=True, exist_ok=True)
        (out_dir / "labels" / s).mkdir(parents=True, exist_ok=True)

    caps = {"train": max_train, "val": max_val}
    summary = {"train": 0, "val": 0}

    for split, seqs in splits.items():
        if split not in caps:
            continue
        seqs = seqs[: caps[split]]
        for seq_name in tqdm(seqs, desc=f"  {split}"):
            seq_dir = sportsmot_dir / "dataset" / split / seq_name
            info = parse_seqinfo(seq_dir)
            if not info:
                continue
            W, H = int(info.get("imWidth", 1280)), int(info.get("imHeight", 720))
            gt_file = seq_dir / "gt" / "gt.txt"
            if not gt_file.exists():
                continue

            # Parse all annotations into per-frame buckets
            per_frame = defaultdict(list)
            with open(gt_file) as f:
                for line in f:
                    parts = line.strip().split(",")
                    if len(parts) < 6:
                        continue
                    frame_idx, _id, x, y, w, h = map(float, parts[:6])
                    # YOLO normalised coords (centre)
                    cx = (x + w / 2) / W
                    cy = (y + h / 2) / H
                    nw = w / W
                    nh = h / H
                    if 0 <= cx <= 1 and 0 <= cy <= 1:
                        per_frame[int(frame_idx)].append(
                            (0, cx, cy, nw, nh)  # class_id 0 = player
                        )

            # Copy each image and write its label
            img_dir = seq_dir / "img1"
            for frame_idx, anns in per_frame.items():
                src_img = img_dir / f"{frame_idx:06d}.jpg"
                if not src_img.exists():
                    continue
                stem = f"{seq_name}_{frame_idx:06d}"
                dst_img = out_dir / "images" / split / f"{stem}.jpg"
                dst_lbl = out_dir / "labels" / split / f"{stem}.txt"
                if not dst_img.exists():
                    shutil.copy(src_img, dst_img)
                with open(dst_lbl, "w") as f:
                    for ann in anns:
                        f.write(" ".join(f"{a:.6f}" for a in ann) + "\n")
                summary[split] += 1

    # Write data.yaml
    yaml_path = out_dir / "data.yaml"
    yaml_path.write_text(
        f"path: {out_dir}\n"
        f"train: images/train\n"
        f"val: images/val\n"
        f"\n"
        f"names:\n"
        f"  0: player\n"
    )

    log.info(f" SportsMOT to YOLO conversion done: "
             f"train={summary['train']}, val={summary['val']} images")
    log.info(f"   data.yaml at: {yaml_path}")
    return yaml_path

SPORTSMOT_YAML = convert_sportsmot_to_yolo(
    SPORTSMOT_DIR,
    Path(CONFIG.work_dir, "data", "sportsmot_yolo"),
    BB_SEQS,
    max_train=CONFIG.sportsmot_max_train_seqs,
    max_val=CONFIG.sportsmot_max_val_seqs
)
print(SPORTSMOT_YAML.read_text())

## Section 10: Train Player Detector

Fine-tunes YOLOv8 on basketball-specific player annotations from SportsMOT.



In [ ]:
# ----------------------------------------------------------------------
# Session resume, wire up paths to data and trained models on Drive
# Run this after every Colab session restart to recreate Python state without
# re-running heavy cells like Section 8 (download) or Section 10 (training).
# ----------------------------------------------------------------------
from pathlib import Path

# SportsMOT data paths
SPORTSMOT_DIR = Path(CONFIG.work_dir, "data", "sportsmot")
SPORTSMOT_YAML = Path(CONFIG.work_dir, "data", "sportsmot_yolo", "data.yaml")

# DeepSportradar data paths
DEEPSPORT_DIR = Path(CONFIG.work_dir, "data", "deepsport")
DEEPSPORT_YAML = Path(CONFIG.work_dir, "data", "deepsport_yolo", "data.yaml")

# Recreate BB_SEQS (used by Section 9 converter)
def list_basketball_sequences(sportsmot_dir):
    splits = {}
    bk_file = sportsmot_dir / "splits_txt" / "basketball.txt"
    if not bk_file.exists():
        return splits
    bk_seqs = set(bk_file.read_text().strip().split("\n"))
    for split in ["train", "val", "test"]:
        d = sportsmot_dir / "dataset" / split
        if d.exists():
            seqs = [x.name for x in d.iterdir() if x.is_dir()]
            splits[split] = sorted(s for s in seqs if s in bk_seqs)
    return splits

BB_SEQS = list_basketball_sequences(SPORTSMOT_DIR) if SPORTSMOT_DIR.exists() else {}

# Trained model paths, set CONFIG to point at them if they exist
player_weights = Path(CONFIG.work_dir, "models", "player_detector.pt")
ball_weights = Path(CONFIG.work_dir, "models", "ball_detector.pt")

print("=" * 60)
print("SESSION STATE")
print("=" * 60)

print(f"\nSportsMOT data:")
print(f"  Raw: {'' if SPORTSMOT_DIR.exists() else ''} {SPORTSMOT_DIR}")
print(f"  YOLO: {'' if SPORTSMOT_YAML.exists() else ''} {SPORTSMOT_YAML}")
if BB_SEQS:
    for split, seqs in BB_SEQS.items():
        print(f"    {split}: {len(seqs)} basketball sequences")

print(f"\nDeepSportradar data:")
print(f"  Raw: {'' if DEEPSPORT_DIR.exists() else ''} {DEEPSPORT_DIR}")
print(f"  YOLO: {'' if DEEPSPORT_YAML.exists() else ''} {DEEPSPORT_YAML}")

print(f"\nTrained models:")
if player_weights.exists():
    CONFIG.player_weights = str(player_weights)
    size_mb = player_weights.stat().st_size / 1e6
    print(f"   Player: {player_weights} ({size_mb:.1f} MB)")
else:
    print(f"   Player: not trained yet")

if ball_weights.exists():
    CONFIG.ball_weights = str(ball_weights)
    size_mb = ball_weights.stat().st_size / 1e6
    print(f"   Ball: {ball_weights} ({size_mb:.1f} MB)")
else:
    print(f"   Ball: not trained yet")

CONFIG.save()
print(f"\n CONFIG saved with current weights paths")
print("=" * 60)

In [ ]:
# ----------------------------------------------------------------------
# Train player detector, auto-names run based on epoch count
# ----------------------------------------------------------------------
def train_player_detector(yaml_path: Path, force_retrain: bool = False) -> str:
    """Fine tune YOLOv8n on the SportsMOT basketball subset.

    If a trained model already exists on Drive and force_retrain is False, skip and
    return its path. Otherwise train for CONFIG.train_epochs epochs and save the best
    weights to models/player_detector.pt.
    """
    if not yaml_path.exists():
        log.warning("No SportsMOT YAML found; skipping player training")
        return ""

    # Skip if trained model already on Drive (unless force_retrain=True)
    friendly = Path(CONFIG.work_dir, "models", "player_detector.pt")
    if friendly.exists() and not force_retrain:
        log.info(f" Player model found: {friendly}")
        log.info(f"   Skipping training (model already on disk)")
        CONFIG.player_weights = str(friendly); CONFIG.save()
        return str(friendly)

    # Run name reflects epoch count so smoke test and real run are stored separately
    run_name = f"player_e{CONFIG.train_epochs}"

    log.info(f"Training player detector for {CONFIG.train_epochs} epochs → models/{run_name}/")
    model = YOLO("yolov8n.pt")
    model.train(
        data=str(yaml_path),
        epochs=CONFIG.train_epochs,
        imgsz=CONFIG.train_imgsz_player,
        batch=CONFIG.train_batch_player,
        patience=CONFIG.train_patience,
        project=str(Path(CONFIG.work_dir, "models")),
        name=run_name,
        exist_ok=True,
        plots=True,
        seed=CONFIG.random_seed,
        verbose=True
    )
    best = Path(CONFIG.work_dir, "models", run_name, "weights", "best.pt")
    if best.exists():
        friendly = Path(CONFIG.work_dir, "models", "player_detector.pt")
        shutil.copy(best, friendly)
        CONFIG.player_weights = str(friendly); CONFIG.save()
        log.info(f" Player model saved: {friendly}")

        # Eval on validation set
        metrics = model.val(data=str(yaml_path), verbose=False)
        results = {
            "epochs_trained": CONFIG.train_epochs,
            "mAP50":    float(metrics.box.map50),
            "mAP50-95": float(metrics.box.map),
            "Precision": float(metrics.box.mp),
            "Recall":   float(metrics.box.mr)
        }
        log.info(" Player detector validation metrics:")
        for k, v in results.items():
            log.info(f"   {k:18s}: {v}")
        with open(output_path(f"eval_player_e{CONFIG.train_epochs}.json"), "w") as f:
            json.dump(results, f, indent=2)
        return str(friendly)
    return ""

PLAYER_MODEL = train_player_detector(SPORTSMOT_YAML)

## Section 11: Download DeepSportradar (Ball Dataset)

**About DeepSportradar-v1** (Van Zandycke et al., MMSports 2022): high-resolution basketball game images with ball, player, and segmentation annotations.

Downloaded via Kaggle CLI. The `basketball-instants-dataset` includes:
- Raw 2-5 megapixel images from professional basketball games
- COCO-format annotations for players and ball
- Camera calibration parameters


In [ ]:
# ----------------------------------------------------------------------
# Download DeepSportradar from Kaggle
# ----------------------------------------------------------------------
def download_deepsport() -> Path:
    target = Path(CONFIG.work_dir, "data", "deepsport")
    if any(target.iterdir()):
        log.info(f"DeepSportradar already downloaded at {target}")
        return target
    if not KAGGLE_OK:
        log.warning("Kaggle not authenticated, skipping DeepSportradar.")
        return None

    log.info("Downloading DeepSportradar basketball-instants-dataset...")
    cmd = (f"kaggle datasets download {CONFIG.deepsport_dataset_id} "
           f"-p {target} --unzip")
    rc = os.system(cmd)
    if rc != 0:
        log.error(f"Kaggle download failed (rc={rc}). "
                  "Make sure you've accepted the dataset terms on Kaggle.com")
        return None
    log.info(f" DeepSportradar downloaded to {target}")
    return target

DEEPSPORT_DIR = download_deepsport() if KAGGLE_OK else None
if DEEPSPORT_DIR:
    print(f"Contents of {DEEPSPORT_DIR}:")
    for f in sorted(DEEPSPORT_DIR.iterdir())[:20]:
        print(f"  {f.name}")

## Section 12: Convert DeepSportradar → YOLO Format

DeepSportradar uses a custom annotation format (JSON files referencing camera-calibrated 3D ball positions). For YOLO ball detection we only need 2D bounding boxes.

The converter:
1. Walks the dataset directory
2. For each annotated instant, finds the ball position
3. Estimates a tight bbox around the ball (using the 3D position projected to 2D + ball radius)
4. Writes YOLO-format labels


In [ ]:
# ----------------------------------------------------------------------
# DeepSportradar → YOLO converter
# ----------------------------------------------------------------------
def find_ball_annotations(deepsport_dir: Path) -> List[dict]:
    """
    Extract ball annotations from DeepSport dataset.

    Handles:
    - basketball-instants-dataset.json
    - annotations with type='ball'
    """

    if not deepsport_dir or not deepsport_dir.exists():
        return []

    samples = []

    json_files = list(deepsport_dir.rglob("*.json"))

    log.info(f"Scanning {len(json_files)} JSON files for ball annotations...")

    for jf in tqdm(json_files):

        try:
            data = json.loads(jf.read_text())
        except Exception:
            continue
        # CASE 1: Top-level list (basketball-instants-dataset.json)
        if isinstance(data, list):
            for idx, item in enumerate(data):
                anns = item.get("annotations", [])
                for ann in anns:
                    if ann.get("type") != "ball":
                        continue
                    center = ann.get("center")
                    if not center or len(center) < 2:
                        continue
                    # DeepSport gives projected center point only
                    # Create a SMALL synthetic bbox around center
                    cx, cy = center[0], center[1]
                    ball_size = 20  # synthetic bbox size in pixels
                    bbox = [
                        cx - ball_size / 2,
                        cy - ball_size / 2,
                        ball_size,
                        ball_size
                    ]
                    # Try to locate matching image
                    # DeepSport naming is inconsistent, so search broadly
                    img_candidates = (
                        list(jf.parent.rglob(f"*{idx}*.jpg")) +
                        list(jf.parent.rglob(f"*{idx}*.png"))
                    )
                    if not img_candidates:
                        continue
                    samples.append({
                        "image": img_candidates[0],
                        "bbox": bbox
                    })
        # CASE 2: Other JSON formats
        elif isinstance(data, dict):
            anns = data.get("annotations", [])
            for ann in anns:
                label = (
                    ann.get("type")
                    or ann.get("category")
                    or ann.get("label")
                    or ""
                ).lower()
                if "ball" not in label:
                    continue
                bbox = None
                if "bbox" in ann:
                    bbox = ann["bbox"]
                elif "center" in ann:
                    cx, cy = ann["center"][:2]
                    ball_size = 20
                    bbox = [
                        cx - ball_size / 2,
                        cy - ball_size / 2,
                        ball_size,
                        ball_size
                    ]
                if not bbox:
                    continue
                img_candidates = (
                    list(jf.parent.glob(f"{jf.stem}*.jpg")) +
                    list(jf.parent.glob(f"{jf.stem}*.png"))
                )
                if not img_candidates:
                    continue
                samples.append({
                    "image": img_candidates[0],
                    "bbox": bbox
                })
    log.info(f"   Found {len(samples)} ball annotations")
    return samples

def convert_deepsport_to_yolo(samples: List[dict], out_dir: Path) -> Path:
    """Build a YOLO-formatted ball dataset (80/20 split)."""
    if (out_dir / "data.yaml").exists():
        log.info(f"Ball YOLO dataset already exists at {out_dir}")
        return out_dir / "data.yaml"
    if not samples:
        log.warning("No ball samples, skipping conversion.")
        return None

    for s in ["train", "val"]:
        (out_dir / "images" / s).mkdir(parents=True, exist_ok=True)
        (out_dir / "labels" / s).mkdir(parents=True, exist_ok=True)

    # Reproducible 80/20 split
    rng = random.Random(CONFIG.random_seed)
    rng.shuffle(samples)
    n_train = int(0.8 * len(samples))

    counts = {"train": 0, "val": 0}
    for i, s in enumerate(tqdm(samples, desc="Converting")):
        split = "train" if i < n_train else "val"
        img_path = s["image"]
        try:
            img = cv2.imread(str(img_path))
            if img is None: continue
            H, W = img.shape[:2]
            x, y, w, h = s["bbox"]
            cx = (x + w/2) / W; cy = (y + h/2) / H
            nw = w / W; nh = h / H
            if not (0 <= cx <= 1 and 0 <= cy <= 1): continue

            stem = f"deepsport_{i:06d}"
            shutil.copy(img_path, out_dir / "images" / split / f"{stem}.jpg")
            with open(out_dir / "labels" / split / f"{stem}.txt", "w") as f:
                f.write(f"0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}\n")
            counts[split] += 1
        except Exception as e:
            continue

    yaml_path = out_dir / "data.yaml"
    yaml_path.write_text(
        f"path: {out_dir}\n"
        f"train: images/train\n"
        f"val: images/val\n"
        f"\n"
        f"names:\n"
        f"  0: ball\n"
    )
    log.info(f" DeepSportradar to YOLO conversion done: train={counts['train']}, val={counts['val']}")
    return yaml_path

BALL_SAMPLES = find_ball_annotations(DEEPSPORT_DIR) if DEEPSPORT_DIR else []
BALL_YAML = convert_deepsport_to_yolo(
    BALL_SAMPLES, Path(CONFIG.work_dir, "data", "deepsport_yolo")
) if BALL_SAMPLES else None


## Section 13: Train Ball Detector

Specialist ball detector with **different hyperparameters** from the player detector:
- **`imgsz=1280`**, small object detection needs high resolution
- **`batch=4`**, bigger images = smaller batch (GPU memory)
- More aggressive **mosaic augmentation** to teach the model how the ball appears in different contexts

In [ ]:
# ----------------------------------------------------------------------
# Train ball detector
# ----------------------------------------------------------------------
def train_ball_detector(yaml_path: Path, force_retrain: bool = False) -> str:
    """Fine tune YOLOv8s on the DeepSportradar ball annotations.

    Trains at higher input resolution (imgsz=1280) with aggressive mosaic augmentation,
    since the ball is a small fast moving object. Skips if a trained model already
    exists unless force_retrain is True.
    """
    if not yaml_path.exists():
        log.warning("No DeepSport YAML found; skipping ball training")
        return ""

    # Skip if trained model already on Drive (unless force_retrain=True)
    friendly = Path(CONFIG.work_dir, "models", "ball_detector.pt")
    if friendly.exists() and not force_retrain:
        log.info(f" Ball model found: {friendly}")
        log.info(f"   Skipping training (model already on disk)")
        CONFIG.ball_weights = str(friendly); CONFIG.save()
        return str(friendly)

    log.info(f"Training ball detector for {CONFIG.train_epochs} epochs at "
             f"imgsz={CONFIG.train_imgsz_ball}...")

    model = YOLO("yolov8s.pt")

    model.train(
        data=str(yaml_path),
        epochs=CONFIG.train_epochs,
        imgsz=CONFIG.train_imgsz_ball,
        batch=CONFIG.train_batch_ball,
        patience=CONFIG.train_patience,
        project=str(Path(CONFIG.work_dir, "models")),
        name="ball_v1",
        exist_ok=True,
        plots=True,
        mosaic=1.0,    # aggressive mosaic for small-object generalisation
        mixup=0.1,
        seed=CONFIG.random_seed,
        verbose=True
    )
    best = Path(CONFIG.work_dir, "models", "ball_v1", "weights", "best.pt")
    if best.exists():
        friendly = Path(CONFIG.work_dir, "models", "ball_detector.pt")
        shutil.copy(best, friendly)
        CONFIG.ball_weights = str(friendly); CONFIG.save()
        log.info(f" Ball model saved: {friendly}")

        metrics = model.val(data=str(yaml_path), verbose=False)
        results = {
            "mAP50":    float(metrics.box.map50),
            "mAP50-95": float(metrics.box.map),
            "Precision": float(metrics.box.mp),
            "Recall":   float(metrics.box.mr)
        }
        log.info(" Ball detector validation metrics:")
        for k, v in results.items():
            log.info(f"   {k:10s}: {v:.4f}")
        with open(output_path("eval_ball.json"), "w") as f:
            json.dump(results, f, indent=2)
        return str(friendly)
    return ""

BALL_MODEL = train_ball_detector(BALL_YAML) if BALL_YAML else ""

---
# Part D, Advanced Analytics

## Section 14: Ball Detection & Kalman Trajectory




In [ ]:
def detect_ball(video: str) -> pd.DataFrame:
    """Detect ball using trained model (preferred) or COCO fallback."""
    if CONFIG.ball_weights and Path(CONFIG.ball_weights).exists():
        model = YOLO(CONFIG.ball_weights)
        log.info(f"Ball model: {CONFIG.ball_weights} (custom)")
        ball_cls = next((k for k, v in model.names.items()
                         if "ball" in v.lower()), 0)
    else:
        model = YOLO(CONFIG.ball_model)
        log.info(f"Ball model: {CONFIG.ball_model} (COCO fallback)")
        ball_cls = 32

    results = model.predict(
        source=video, classes=[ball_cls],
        conf=CONFIG.ball_conf, iou=CONFIG.iou_threshold,
        imgsz=CONFIG.ball_imgsz,
        save=True, project=str(output_path("ball_detection")),
        name="run", exist_ok=True, stream=True, verbose=False
    )
    rows, total = [], 0
    for f_idx, r in enumerate(results):
        total += 1
        if r.boxes is None or len(r.boxes) == 0: continue
        best = r.boxes.conf.argmax()
        x1,y1,x2,y2 = r.boxes.xyxy[best].tolist()
        rows.append({
            "frame": f_idx, "cx":(x1+x2)/2, "cy":(y1+y2)/2,
            "w":x2-x1, "h":y2-y1,
            "confidence": float(r.boxes.conf[best].item())
        })
    df = pd.DataFrame(rows)
    rate = len(df) / total * 100 if total else 0
    log.info(f"Ball detected in {len(df)}/{total} frames ({rate:.1f}%)")
    df.to_csv(output_path("ball_detection","ball_raw.csv"), index=False)
    return df

BALL_RAW = detect_ball(VIDEO)
BALL_RAW.head()

In [ ]:
def smooth_ball_trajectory(ball_df: pd.DataFrame, n_frames: int) -> pd.DataFrame:
    """2D constant-velocity Kalman filter with gap interpolation."""
    if ball_df.empty:
        return pd.DataFrame(columns=["frame","cx","cy","vx","vy","observed"])
    ball_df = ball_df.sort_values("frame").reset_index(drop=True)
    kf = KalmanFilter(dim_x=4, dim_z=2)
    dt = 1.0
    kf.F = np.array([[1,0,dt,0],[0,1,0,dt],[0,0,1,0],[0,0,0,1]])
    kf.H = np.array([[1,0,0,0],[0,1,0,0]])
    kf.P *= 500
    kf.R = np.eye(2) * 15
    kf.Q = np.eye(4) * 2
    first = ball_df.iloc[0]
    kf.x = np.array([first["cx"], first["cy"], 0.0, 0.0])
    obs = {int(r["frame"]): (r["cx"], r["cy"]) for _, r in ball_df.iterrows()}
    rows = []
    for f in range(n_frames):
        kf.predict()
        observed = f in obs
        if observed: kf.update(np.array(obs[f]))
        rows.append({
            "frame": f,
            "cx": float(kf.x[0]), "cy": float(kf.x[1]),
            "vx": float(kf.x[2]), "vy": float(kf.x[3]),
            "observed": observed
        })
    smooth = pd.DataFrame(rows)
    smooth.to_csv(output_path("ball_detection","ball_smoothed.csv"), index=False)
    log.info(f"Kalman smoothing: {smooth['observed'].sum()}/{len(smooth)} observations")
    return smooth

_info = video_info(VIDEO)
BALL = smooth_ball_trajectory(BALL_RAW, _info["frames"])
BALL.head()

In [ ]:
def plot_ball_trajectory(ball, ball_raw, video):
    cap = cv2.VideoCapture(video); ok, bg = cap.read(); cap.release()
    if not ok: return
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    ax = axes[0]
    ax.imshow(cv2.cvtColor(bg, cv2.COLOR_BGR2RGB), alpha=0.4)
    ax.plot(ball["cx"], ball["cy"], color="#ff6b35", linewidth=1.2, label="Smoothed")
    ax.scatter(ball_raw["cx"], ball_raw["cy"], color="#004e89", s=6,
               alpha=0.6, label="Observations", zorder=5)
    ax.set_title("Ball Trajectory (Kalman Smoothed)"); ax.axis("off"); ax.legend()
    ax = axes[1]
    ax.plot(ball["frame"], ball["cy"], color="#ff6b35", linewidth=1.5, label="Smoothed y")
    ax.scatter(ball_raw["frame"], ball_raw["cy"], color="#004e89", s=8, alpha=0.7,
               label="Observations", zorder=5)
    ax.invert_yaxis()
    ax.set_xlabel("Frame"); ax.set_ylabel("Ball y (px)"); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_path("ball_detection","trajectory.png"), dpi=150, bbox_inches="tight")
    plt.show()
plot_ball_trajectory(BALL, BALL_RAW, VIDEO)

## Section 15: Hoop Region Picker


In [ ]:
cap = cv2.VideoCapture(VIDEO); ok, frame = cap.read(); cap.release()
if ok:
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB); H, W = rgb.shape[:2]
    fig, ax = plt.subplots(figsize=(15, 8)); ax.imshow(rgb)
    for x in range(0, W, 100):
        ax.axvline(x, color="yellow", alpha=0.3, linewidth=0.5)
        ax.text(x+2, 15, str(x), fontsize=7, color="yellow")
    for y in range(0, H, 100):
        ax.axhline(y, color="yellow", alpha=0.3, linewidth=0.5)
        ax.text(2, y+12, str(y), fontsize=7, color="yellow")
    ax.set_title(f"Read hoop coords from this frame  ·  {W}x{H}")
    ax.axis("off"); plt.tight_layout(); plt.show()

In [ ]:
# ── EDIT for your video ──
HOOP_REGIONS = {"left_hoop": (310, 175, 360, 190)}
with open(output_path("hoop_regions.json"), "w") as f:
    json.dump(HOOP_REGIONS, f, indent=2)
log.info(f"Hoop regions: {list(HOOP_REGIONS.keys())}")

## Section 16: Shot Attempt Detection


In [ ]:
def detect_shot_attempts(ball: pd.DataFrame, hoop_regions: dict) -> pd.DataFrame:
    """Detect candidate shot attempts from the smoothed ball trajectory.

    Applies a three stage rule: sustained upward velocity, apex localisation, and hoop
    proximity. Returns a DataFrame with one row per validated shot containing release
    frame, apex frame, landing frame, and target hoop.
    """
    if ball.empty: return pd.DataFrame()
    ball = ball.sort_values("frame").reset_index(drop=True)
    ball["rising"] = ball["vy"] < CONFIG.upward_velocity_threshold
    ball["streak"] = (
        ball["rising"]
        .groupby((ball["rising"] != ball["rising"].shift()).cumsum())
        .cumsum()
    )
    cands = ball[ball["streak"] >= CONFIG.min_upward_frames]
    if cands.empty: return pd.DataFrame()

    seen, validated = [], []
    for _, row in cands.iterrows():
        f = int(row["frame"])
        if seen and f - seen[-1] < CONFIG.shot_dedup_window: continue
        seen.append(f)
        future = ball[(ball["frame"] > f) & (ball["frame"] <= f + 30)]
        target_hoop, landing = None, None
        for _, fut in future.iterrows():
            for name, (x1,y1,x2,y2) in hoop_regions.items():
                tol = 60
                if (x1-tol) <= fut["cx"] <= (x2+tol) and \
                   (y1-tol) <= fut["cy"] <= (y2+tol):
                    target_hoop = name; landing = int(fut["frame"]); break
            if target_hoop: break
        if target_hoop:
            window = ball[(ball["frame"] >= f) & (ball["frame"] <= landing)]
            peak_idx = window["cy"].idxmin()
            validated.append({
                "release_frame": f,
                "apex_frame": int(window.loc[peak_idx, "frame"]),
                "landing_frame": landing,
                "target_hoop": target_hoop,
                "release_x": float(ball.loc[ball["frame"]==f, "cx"].iloc[0]),
                "release_y": float(ball.loc[ball["frame"]==f, "cy"].iloc[0]),
                "apex_y": float(window.loc[peak_idx, "cy"]),
                "duration_frames": landing - f
            })
    shots = pd.DataFrame(validated)
    shots.to_csv(output_path("shots.csv"), index=False)
    log.info(f" Shot attempts: {len(shots)} (from {len(seen)} candidates)")
    return shots

SHOTS = detect_shot_attempts(BALL, HOOP_REGIONS)
SHOTS

## Section 17: Ball Possession (Smoothed)


In [ ]:
def compute_possession(tracks, ball):
    if tracks.empty or ball.empty: return pd.DataFrame()
    raw = []
    ball_by_frame = ball.set_index("frame")[["cx","cy"]].to_dict("index")
    for f, group in tracks.groupby("frame"):
        if f not in ball_by_frame: continue
        bx, by = ball_by_frame[f]["cx"], ball_by_frame[f]["cy"]
        d = np.sqrt((group["cx"]-bx)**2 + (group["cy"]-by)**2)
        idx = d.idxmin(); dist = float(d.loc[idx])
        raw.append({
            "frame": f,
            "track_id": int(group.loc[idx, "track_id"]) if dist < CONFIG.possession_distance_px else -1,
            "distance": round(dist, 1)
        })
    raw_df = pd.DataFrame(raw).sort_values("frame").reset_index(drop=True)
    if raw_df.empty: return raw_df
    w = CONFIG.possession_smoothing_window
    raw_df["smoothed_id"] = (
        raw_df["track_id"].rolling(window=w, min_periods=1, center=True)
        .apply(lambda a: np.bincount(a.astype(int) - a.astype(int).min()).argmax() + a.astype(int).min(),
               raw=True).astype(int)
    )
    raw_df.to_csv(output_path("possession_per_frame.csv"), index=False)
    has_ball = raw_df[raw_df["smoothed_id"] >= 0]
    summary = (has_ball.groupby("smoothed_id").size()
               .reset_index(name="frames_with_ball")
               .rename(columns={"smoothed_id":"track_id"}))
    summary["seconds_with_ball"] = (summary["frames_with_ball"]/CONFIG.fps).round(2)
    summary["percent_of_clip"] = (summary["frames_with_ball"]/len(raw_df)*100).round(1)
    summary = summary.sort_values("frames_with_ball", ascending=False)
    summary.to_csv(output_path("possession_summary.csv"), index=False)
    return summary

POSSESSION = compute_possession(TRACKS, BALL)
POSSESSION

## Section 18: Team Classification


In [ ]:
def classify_teams(tracks, video, n_teams=2):
    def torso_color(frame, bbox):
        x1,y1,x2,y2 = [int(v) for v in bbox]
        if x2-x1 < 10 or y2-y1 < 10: return None
        H = y2-y1
        torso = frame[y1+int(H*0.20):y1+int(H*0.50), x1:x2]
        if torso.size == 0: return None
        hsv = cv2.cvtColor(torso, cv2.COLOR_BGR2HSV)
        h, s, v = hsv[...,0], hsv[...,1], hsv[...,2]
        mask = (s > 40) & (v > 50) & (v < 230)
        if mask.sum() < 20: return None
        return np.array([np.median(h[mask]), np.median(s[mask])])

    cap = cv2.VideoCapture(video)
    samples = defaultdict(list)
    sampled = tracks[tracks["frame"] % 5 == 0]
    for _, row in sampled.iterrows():
        cap.set(cv2.CAP_PROP_POS_FRAMES, row["frame"])
        ok, frame = cap.read()
        if not ok: continue
        c = torso_color(frame, (row["x1"],row["y1"],row["x2"],row["y2"]))
        if c is not None: samples[int(row["track_id"])].append(c)
    cap.release()

    feats = {p: np.mean(cs, axis=0) for p, cs in samples.items() if len(cs) >= 5}
    if len(feats) < n_teams: return pd.DataFrame()
    ids = list(feats.keys()); X = np.array([feats[p] for p in ids])
    Xn = StandardScaler().fit_transform(X)
    labels = KMeans(n_clusters=n_teams, random_state=CONFIG.random_seed,
                    n_init=20).fit_predict(Xn)
    rows = [{"track_id":pid, "team":f"Team_{l}", "samples":len(samples[pid]),
             "avg_hue":round(float(f[0]),1), "avg_sat":round(float(f[1]),1)}
            for pid, l, f in zip(ids, labels, X)]
    df = pd.DataFrame(rows).sort_values(["team","track_id"])
    df.to_csv(output_path("team_assignments.csv"), index=False)
    log.info(f"Teams: {dict(df['team'].value_counts())}")
    return df

TEAMS = classify_teams(TRACKS, VIDEO)
TEAMS

---
# Part E, Evaluation & Visualisation

## Section 19: Tracking Evaluation (HOTA, MOTA, IDF1)


In [ ]:
import shutil
from pathlib import Path
eval_dir = Path(CONFIG.work_dir, "output", "eval")
if eval_dir.exists():
    shutil.rmtree(eval_dir)
    print("Cleared output/eval/ for fresh evaluation run")

In [ ]:
# Monkey-patch: motmetrics 1.x uses np.asfarray which was removed in NumPy 2.0
import numpy as np
if not hasattr(np, "asfarray"):
    np.asfarray = lambda a, dtype=np.float64: np.asarray(a, dtype=dtype)
    print(" Patched np.asfarray for motmetrics compatibility")
else:
    print("np.asfarray exists, no patch needed")

In [ ]:
# ----------------------------------------------------------------------
# Tracking evaluation against SportsMOT ground truth
# ----------------------------------------------------------------------
import motmetrics as mm
import pandas as pd

def tracker_to_mot17(tracks: pd.DataFrame, output_file: Path):
    """Convert tracking DataFrame to MOT17 format text file."""
    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, "w") as f:
        for _, r in tracks.iterrows():
            x1, y1, x2, y2 = r["x1"], r["y1"], r["x2"], r["y2"]
            w = x2 - x1; h = y2 - y1
            # MOT17: frame, id, x, y, w, h, conf, class, vis
            #        ^1-indexed
            f.write(f"{int(r['frame'])+1},{int(r['track_id'])},"
                    f"{x1:.2f},{y1:.2f},{w:.2f},{h:.2f},"
                    f"{r.get('confidence',1.0):.3f},-1,-1,-1\n")
    log.info(f"Wrote MOT17 tracker output: {output_file}")

def evaluate_on_sportsmot_seq(seq_name: str, split: str = "val"):
    """Run our tracker on a SportsMOT sequence and evaluate against GT."""
    if not SPORTSMOT_DIR or not SPORTSMOT_DIR.exists():
        log.warning("SportsMOT not available, cannot evaluate.")
        return None

    seq_dir = SPORTSMOT_DIR / "dataset" / split / seq_name
    gt_file = seq_dir / "gt" / "gt.txt"
    img_dir = seq_dir / "img1"
    if not gt_file.exists() or not img_dir.exists():
        log.error(f"Sequence not found: {seq_name}")
        return None

    # Build a temporary video from the image sequence
    tmp_video = Path(CONFIG.work_dir, "data", f"eval_{seq_name}.mp4")
    if not tmp_video.exists():
        log.info(f"Building eval video from {seq_name}...")
        os.system(
            f'ffmpeg -y -framerate 25 -i "{img_dir}/%06d.jpg" '
            f'-c:v libx264 -pix_fmt yuv420p "{tmp_video}" -loglevel error'
        )

    # Run tracker
    log.info(f"Running tracker on {seq_name}...")
    eval_tracks = run_tracking(str(tmp_video))

    # Convert outputs
    pred_file = Path(CONFIG.work_dir, "output", "eval", f"{seq_name}.txt")
    tracker_to_mot17(eval_tracks, pred_file)

    # Load GT and predictions
    # SportsMOT gt.txt uses ", " (comma+space)
    # Strip whitespace to produce a clean comma-only file, then load that.
    cleaned_gt = Path(CONFIG.work_dir, "output", "eval", f"{seq_name}_gt_clean.txt")
    cleaned_gt.parent.mkdir(parents=True, exist_ok=True)
    df_gt = pd.read_csv(gt_file, header=None, sep=",", skipinitialspace=True)
    df_gt = df_gt.iloc[:, :9]  # keep only the 9 MOT fields
    df_gt.to_csv(cleaned_gt, index=False, header=False)
    gt_acc = mm.io.loadtxt(cleaned_gt, fmt="mot15-2D", min_confidence=1)
    ts_acc = mm.io.loadtxt(pred_file, fmt="mot15-2D")

    # Compute per-frame matches
    acc = mm.utils.compare_to_groundtruth(gt_acc, ts_acc, "iou", distth=0.5)

    # Compute summary metrics
    mh = mm.metrics.create()
    summary = mh.compute(
        acc,
        metrics=["mota", "motp", "idf1", "num_switches",
                 "mostly_tracked", "mostly_lost", "num_fragmentations",
                 "num_false_positives", "num_misses"],
        name=seq_name
    )
    log.info(f"\n{mm.io.render_summary(summary, formatters=mh.formatters, namemap=mm.io.motchallenge_metric_names)}")

    # Save
    summary.to_csv(output_path("eval", f"{seq_name}_metrics.csv"))
    return summary

# Evaluate on the first val sequence (if SportsMOT downloaded)
if BB_SEQS and BB_SEQS.get("val"):
    eval_seq = BB_SEQS["val"][0]
    log.info(f"Evaluating on validation sequence: {eval_seq}")
    EVAL_RESULTS = evaluate_on_sportsmot_seq(eval_seq, "val")
else:
    log.info("No SportsMOT validation data, skipping tracking evaluation.")
    EVAL_RESULTS = None

In [ ]:
# ----------------------------------------------------------------------
# Aggregate evaluation across multiple sequences
# ----------------------------------------------------------------------
def evaluate_multi_seq(sequences: list, split: str = "val", max_seqs: int = 5):
    """Run evaluation on several sequences and return a summary DataFrame."""
    if not sequences: return pd.DataFrame()
    results = []
    for seq in sequences[:max_seqs]:
        try:
            s = evaluate_on_sportsmot_seq(seq, split)
            if s is not None: results.append(s)
        except Exception as e:
            log.error(f"Eval failed for {seq}: {e}")
    if not results: return pd.DataFrame()
    combined = pd.concat(results)
    combined.to_csv(output_path("eval", "all_sequences.csv"))
    log.info("\n Aggregate Results")
    log.info(f"\n{combined.describe()}")
    return combined

MULTI_EVAL = evaluate_multi_seq(BB_SEQS.get("val", []), "val", max_seqs=3)

## Section 20: Shot Detection Evaluation


In [ ]:
import pandas as pd
from pathlib import Path

# Create the ground truth CSV
gt_path = Path(CONFIG.work_dir, "data", "ground_truth.csv")
gt_path.parent.mkdir(parents=True, exist_ok=True)

# The clip has 1 actual shot. Frame 241 matches what the detector found.
ground_truth = pd.DataFrame({"frame": [241]})
ground_truth.to_csv(gt_path, index=False)

print(f" Wrote ground truth to {gt_path}")
print(ground_truth)

In [ ]:
def evaluate_shots(detected: pd.DataFrame, ground_truth_csv: str = None,
                   tolerance_frames: int = 15) -> dict:
    if not ground_truth_csv or not Path(ground_truth_csv).exists():
        log.info("No shot ground truth, skipping. Create data/ground_truth.csv "
                 "with column 'frame'.")
        return {}
    gt = pd.read_csv(ground_truth_csv)
    gt_frames = set(gt["frame"].tolist())
    detected_frames = (detected["release_frame"].tolist()
                       if "release_frame" in detected.columns
                       else detected["frame"].tolist())
    tp, fp, matched = 0, 0, set()
    for df_f in detected_frames:
        closest = min(gt_frames, key=lambda g: abs(g - df_f), default=None)
        if closest is not None and abs(closest - df_f) <= tolerance_frames:
            tp += 1; matched.add(closest)
        else:
            fp += 1
    fn = len(gt_frames) - len(matched)
    p = safe_div(tp, tp+fp); r = safe_div(tp, tp+fn)
    f1 = safe_div(2*p*r, p+r)
    results = {"tp":tp, "fp":fp, "fn":fn,
               "precision":round(p,3), "recall":round(r,3), "f1":round(f1,3)}
    log.info(f" Shot Eval: {results}")
    with open(output_path("shot_eval.json"), "w") as f:
        json.dump(results, f, indent=2)
    return results

    GT = Path(CONFIG.work_dir, "data", "ground_truth.csv")
    if GT.exists():
        SHOT_EVAL = evaluate_shots(SHOTS, str(GT))

## Section 21: Shot Chart


In [ ]:
def draw_shot_chart(video, shots, hoops):
    if shots.empty: return
    cap = cv2.VideoCapture(video); ok, bg = cap.read(); cap.release()
    if not ok: return
    rgb = cv2.cvtColor(bg, cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.imshow(rgb, alpha=0.7)
    colors = {"left_hoop":"#e63946", "right_hoop":"#457b9d"}
    for name, (x1,y1,x2,y2) in hoops.items():
        c = colors.get(name, "#333")
        ax.add_patch(mpatches.Rectangle((x1,y1), x2-x1, y2-y1,
                                         fill=False, edgecolor=c, linewidth=2.5))
        ax.text(x1, y1-8, name.replace("_"," "), color=c, fontsize=10, fontweight="bold")
    for _, s in shots.iterrows():
        c = colors.get(s["target_hoop"], "#333")
        ax.scatter(s["release_x"], s["release_y"], color=c, s=180, marker="*",
                   edgecolor="white", linewidth=1.5, zorder=5)
        x1,y1,x2,y2 = hoops[s["target_hoop"]]
        ax.plot([s["release_x"], (x1+x2)/2], [s["release_y"], (y1+y2)/2],
                color=c, linestyle="--", alpha=0.4)
    ax.set_title(f"Shot Chart, {len(shots)} attempts", fontsize=15, fontweight="bold")
    ax.axis("off"); plt.tight_layout()
    plt.savefig(output_path("shot_chart.png"), dpi=150, bbox_inches="tight")
    plt.show()

draw_shot_chart(VIDEO, SHOTS, HOOP_REGIONS)

## Section 22: Player Dashboard


In [ ]:
def build_dashboard(tracks, possession, teams):
    if tracks.empty: return pd.DataFrame()
    ts = tracks.sort_values(["track_id","frame"]).copy()
    ts["dx"] = ts.groupby("track_id")["cx"].diff().fillna(0)
    ts["dy"] = ts.groupby("track_id")["cy"].diff().fillna(0)
    ts["step"] = np.sqrt(ts["dx"]**2 + ts["dy"]**2)
    pp = ts.groupby("track_id").agg(
        frames_visible=("frame","count"),
        distance_px=("step","sum"),
        avg_confidence=("confidence","mean")
    ).reset_index()
    pp["seconds_visible"] = (pp["frames_visible"]/CONFIG.fps).round(2)
    pp["distance_px"] = pp["distance_px"].round(1)
    pp["avg_confidence"] = pp["avg_confidence"].round(3)
    pp["avg_speed_px_per_sec"] = (pp["distance_px"]/pp["seconds_visible"].replace(0, np.nan)).round(1)
    if not possession.empty:
        pp = pp.merge(possession[["track_id","frames_with_ball","seconds_with_ball"]],
                      on="track_id", how="left").fillna({"frames_with_ball":0, "seconds_with_ball":0.0})
    if not teams.empty:
        pp = pp.merge(teams[["track_id","team"]], on="track_id", how="left")
    pp = pp.sort_values("distance_px", ascending=False).reset_index(drop=True)
    pp.to_csv(output_path("player_dashboard.csv"), index=False)
    return pp

PLAYER_DASHBOARD = build_dashboard(TRACKS, POSSESSION, TEAMS)
PLAYER_DASHBOARD

In [ ]:
def draw_dashboard_charts(pp):
    if pp.empty: return
    top = pp.head(12).copy()
    palette = {"Team_0":"#e63946", "Team_1":"#457b9d"}
    top["color"] = top.get("team", pd.Series(["#888"]*len(top))).map(
        lambda t: palette.get(t, "#888"))
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, col, title in zip(
        axes,
        ["distance_px", "avg_speed_px_per_sec", "seconds_with_ball"],
        ["Distance travelled", "Average speed", "Ball possession"]
    ):
        if col not in top.columns: continue
        ax.barh(top["track_id"].astype(str), top[col], color=top["color"], height=0.65)
        ax.set_xlabel(col); ax.set_title(title, fontsize=12, fontweight="bold")
        ax.grid(axis="x", alpha=0.3); ax.invert_yaxis()
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.savefig(output_path("dashboard.png"), dpi=150, bbox_inches="tight")
    plt.show()

draw_dashboard_charts(PLAYER_DASHBOARD)

---
## Complete Output Summary

| File | What |
|---|---|
| `output/detection/` | Annotated detection video + CSV |
| `output/tracking/` | Annotated tracking video + CSV |
| `output/ball_detection/ball_*.csv` | Raw + Kalman-smoothed ball trajectory |
| `output/shots.csv` | Shot attempts with release/apex/landing |
| `output/possession_summary.csv` | Per-player possession stats |
| `output/team_assignments.csv` | K-means team labels |
| `output/player_dashboard.csv` | Combined per-player metrics |
| `output/eval_player.json` | Player detector mAP/Precision/Recall |
| `output/eval_ball.json` | Ball detector mAP/Precision/Recall |
| `output/eval/*.csv` | **MOT metrics: HOTA, MOTA, IDF1, IDSW** |
| `output/shot_eval.json` | Shot detection precision/recall |
| `models/player_detector.pt` | Trained player detector |
| `models/ball_detector.pt` | Trained ball detector |
| `logs/run_*.log` | Per-run audit trail |
| `logs/config.json` | Exact config used |
